# ClassCounter Examples

This notebook demonstrates how to use `classcounter` to count occurrences of class values in arrays and tensors.

In [1]:
import numpy as np

from classcounter import count_classes

## Basic Usage

Pass a numpy array to `count_classes` and get back a dict mapping each unique value to its count.

In [2]:
rng = np.random.default_rng(42)
arr = rng.integers(0, 5, size=(100, 100), dtype=np.int32)

count_classes(arr)

{0: 2022, 1: 1993, 2: 2077, 3: 1967, 4: 1941}

## Name Mapping

Use the `names` parameter to map integer class IDs to human-readable labels.

In [3]:
names = {0: "water", 1: "forest", 2: "urban", 3: "crop", 4: "bare"}

count_classes(arr, names=names)

{'water': 2022, 'forest': 1993, 'urban': 2077, 'crop': 1967, 'bare': 1941}

Unmapped classes are preserved with their integer key:

In [4]:
# Only map some classes -- class 3 and 4 appear as raw integers
partial_names = {0: "water", 1: "forest", 2: "urban"}

count_classes(arr, names=partial_names)

/tmp/ipykernel_4084387/923967057.py:4: UserWarning: Class 3 found in data but missing from names mapping. Using integer value as key.
  count_classes(arr, names=partial_names)
/tmp/ipykernel_4084387/923967057.py:4: UserWarning: Class 4 found in data but missing from names mapping. Using integer value as key.
  count_classes(arr, names=partial_names)


{'water': 2022, 'forest': 1993, 'urban': 2077, 3: 1967, 4: 1941}

## Percentage Output

Set `percent=True` to get results as percentages (0–100) instead of raw counts.

In [5]:
count_classes(arr, percent=True)

{0: 20.22, 1: 19.93, 2: 20.77, 3: 19.67, 4: 19.41}

Combines with name mapping:

In [6]:
count_classes(arr, names=names, percent=True)

{'water': 20.22, 'forest': 19.93, 'urban': 20.77, 'crop': 19.67, 'bare': 19.41}

## PyTorch Tensors

Requires `pip install classcounter[torch]`. Pass a torch tensor and the torch backend is used automatically.

In [7]:
import torch

tensor = torch.randint(0, 5, (100, 100))
count_classes(tensor)

{0: 1977, 1: 1978, 2: 1969, 3: 2012, 4: 2064}

## Numba Acceleration

If `pip install classcounter[numba]` is installed, numpy arrays are automatically counted with a JIT-compiled loop — no code changes needed.

In [8]:
# With numba installed, this automatically uses the numba backend
count_classes(arr)

{0: 2022, 1: 1993, 2: 2077, 3: 1967, 4: 1941}

## Large Array Performance

For large arrays, install `classcounter[numba]` or use PyTorch for best performance.

In [12]:
big = rng.integers(0, 10, size=(20000, 20000), dtype=np.int32)
print(f"Elements: {big.size:,}")

%timeit count_classes(big)

Elements: 400,000,000
408 ms ± 14.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## GPU Acceleration with PyTorch

When a CUDA GPU is available, placing the tensor on the GPU before calling `count_classes` can be significantly faster for large data. The torch backend runs `bincount` directly on the device, avoiding CPU overhead.

In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

big_tensor = torch.randint(0, 10, (20000, 20000), device=device)

# CPU baseline
big_cpu = big_tensor.cpu()
%timeit count_classes(big_cpu)

# GPU (or same CPU if no GPU available)
%timeit count_classes(big_tensor)

Using device: cuda
736 ms ± 50 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
40.6 ms ± 2.02 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Loading a GeoTIFF

Requires `pip install classcounter[geo]`. Pass a file path directly to count classes in a raster.

In [16]:
# count_classes("land_cover.tif", names={1: "forest", 2: "water", 3: "urban"})